# Lab 1: Advanced Medical Image Pre-Processing Methods
**Biomedical Engineering Degree — Universidad Rey Juan Carlos (URJC)**  
*Course: Medical Image Analysis (Academic Year 2026/2027)*  

---  
### 👥 Group Members:
- Student 1:
- Student 2:

---  
### 🎯 Lab Objectives:
1. **Load and characterize** two distinct grayscale clinical images.
2. **Add noise**: Gaussian and Impulsive (Salt & Pepper) with 3 intensity levels (low, medium, high).
3. **Standard smoothing methods**: Mean filter, Gaussian filter, and Median filter.
4. **Non-Local Means (NLM)**: Parameter tuning and comparison against standard methods.
5. **Anisotropic diffusion (Perona & Malik)**: Edge-preserving smoothing and parameter analysis.
6. **Advanced state-of-the-art filtering**: Literature method exploration, implementation, and benchmarking.
7. **Consolidate metrics (PSNR, SSIM)** and export high-resolution figures for the final report (max. 10 pages).


In [ ]:
# =============================================================================
# SETUP AND IMPORTS
# =============================================================================
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Ensure access to src/ package from both /notebooks and project root
PROJECT_ROOT = Path("..").resolve() if Path("../src").exists() else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Custom modular functions implemented in src/
from src.utils import load_gray, save_figure
from src.noise import add_gaussian_noise, add_impulse_noise, GAUSS_LEVELS, IMPULSE_LEVELS
from src.metrics import compute_psnr, compute_ssim, evaluate_quality
from src.filters_standard import filter_mean, filter_gaussian, filter_median
from src.filters_nlm import filter_nlm
from src.filters_anisotropic import filter_perona_malik
from src.filters_advanced import filter_bilateral, filter_total_variation

DATA_DIR = Path("../data/raw") if Path("../data/raw").exists() else Path("data/raw")
print("✓ Modules loaded successfully. Data directory:", DATA_DIR)


## 0. Loading and Visualizing Original Images
Two grayscale clinical modalities with distinct anatomical properties are selected:
- **Image 1**: T1-weighted Brain MRI (`Brain_MRI_131749_T1.png`).
- **Image 2**: Posteroanterior (PA) Chest X-ray (`Chest_Xray_PA_3-8-2010.png`).


In [ ]:
# Load the two selected grayscale clinical images
img1 = load_gray(DATA_DIR / "Brain_MRI_131749_T1.png")
img2 = load_gray(DATA_DIR / "Chest_Xray_PA_3-8-2010.png")

images = [img1, img2]
titles = ["Brain MRI (T1)", "Chest X-ray (PA)"]

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for col, (im, t) in enumerate(zip(images, titles)):
    # Top row: Grayscale image [0.0, 1.0]
    axes[0, col].imshow(im, cmap="gray", vmin=0, vmax=1)
    axes[0, col].set_title(t)
    axes[0, col].axis("off")

    # Bottom row: Intensity histogram
    axes[1, col].hist(im.ravel(), bins=256, range=(0, 1), color="gray")
    axes[1, col].set_xlabel("Intensity")
    axes[1, col].set_ylabel("Number of pixels")
    axes[1, col].set_title(f"Histogram (min={im.min():.2f}, max={im.max():.2f})")

plt.tight_layout()
# Automatically save high-resolution figure for the final report
save_figure(fig, "00_original_images_histograms.png")
plt.show()


## 1. Noise Addition
As stated in the assignment instructions, two major noise types are corrupted at **3 intensity levels** (low, medium, high):
1. **Gaussian Noise**: Additive zero-mean noise with standard deviation $\sigma$.
2. **Impulse Noise (Salt & Pepper)**: Corrupted pixels set to extreme dynamic values (0.0 or 1.0).


In [ ]:
# =============================================================================
# 1.1 GAUSSIAN NOISE (Levels: low=0.01, medium=0.08, high=0.20)
# =============================================================================
def show_gaussian(img, name, seed, filename):
    rng = np.random.default_rng(seed)        # Fixed seed for reproducibility
    versions = {"Original": img}
    for level in ["low", "medium", "high"]:
        sigma = GAUSS_LEVELS[level]
        versions[f"sigma={sigma} ({level})"] = add_gaussian_noise(img, sigma, rng)

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for col, (title, im) in enumerate(versions.items()):
        axes[0, col].imshow(im, cmap="gray", vmin=0, vmax=1)
        p = "" if title == "Original" else f"\nPSNR={compute_psnr(img, im):.1f} dB"
        axes[0, col].set_title(title + p)
        axes[0, col].axis("off")

        axes[1, col].hist(im.ravel(), bins=256, range=(0, 1), color="gray", log=True)
        axes[1, col].set_xlabel("Intensity")
        axes[1, col].set_ylabel("Number of pixels (log)")
    fig.suptitle(f"Gaussian Noise: {name}")
    plt.tight_layout()
    save_figure(fig, filename)
    plt.show()

show_gaussian(img1, "Brain MRI", seed=100, filename="01_gaussian_noise_mri.png")
show_gaussian(img2, "Chest X-ray", seed=101, filename="01_gaussian_noise_xray.png")


### 1.2 Impulse Noise (Salt and Pepper)
> **YOUR TURN (TODO)**:  
> Implement the `show_impulse(...)` function following the pattern above:
> - Iterate over `IMPULSE_LEVELS` (`'low'`, `'medium'`, `'high'`).
> - Use `add_impulse_noise(img, amount=..., rng=rng)` from `src.noise`.
> - Compute degradation PSNR using `compute_psnr(img, im)`.
> - Save the figures using `save_figure(fig, "02_impulse_noise_...")`.


In [ ]:
# TODO: Define show_impulse and execute it for img1 and img2
def show_impulse(img, name, seed, filename):
    # [Write your implementation here]
    pass

# TODO: Call show_impulse for both images
# show_impulse(img1, "Brain MRI", seed=200, filename="02_impulse_noise_mri.png")
# show_impulse(img2, "Chest X-ray", seed=201, filename="02_impulse_noise_xray.png")


## 2. Standard Smoothing Methods
In this section, filter both images corrupted with the three levels of Gaussian and impulse noise using:
- **Mean filter**: `filter_mean(img, size=...)`
- **Gaussian filter**: `filter_gaussian(img, sigma=...)`
- **Median filter**: `filter_median(img, size=...)`

> **Assignment Requirement**: Determine the filter parameters that provide the best performance (highest PSNR and SSIM) for each case. Justify parameter selection and analyze edge blurring vs noise suppression.


### 2.1 Filtering Gaussian Noise with Standard Methods
> **YOUR TURN (TODO)**:  
> 1. Select a noisy image (e.g. `img1` with medium/high Gaussian noise).
> 2. Test different kernel sizes for Mean (e.g. 3, 5, 7) and $\sigma$ for Gaussian (e.g. 0.5, 1.0, 1.5, 2.0).
> 3. Compare with Median filter.
> 4. Compute metrics using `evaluate_quality(original, filtered)`.
> 5. Plot visual comparisons and save figure: `save_figure(fig, "03_standard_gaussian_noise.png")`.


In [ ]:
# TODO: Experiments with standard filters on Gaussian noise
# [Call filter_mean, filter_gaussian, filter_median here]


### 2.2 Filtering Impulse Noise with Standard Methods
> **YOUR TURN (TODO)**:  
> 1. Take the image corrupted with impulse noise.
> 2. Apply mean, Gaussian, and median filters (test window sizes $3\times3$ and $5\times5$).
> 3. Observe why median filter is optimal for impulse noise compared to linear averaging.
> 4. Save the figure: `save_figure(fig, "04_standard_impulse_noise.png")`.


In [ ]:
# TODO: Experiments with standard filters on impulse noise
# [Call standard filters here and compare]


## 3. Non-Local Means (NLM) Filtering
The Non-Local Means algorithm filters the image by weighted averages of pixels based on the **similarity of their local patch neighborhoods**, rather than spatial proximity alone.

> **Assignment Requirement**:  
> - Evaluate NLM with both images, both noise types, and all 3 intensity levels.
> - Tune the key parameters of `filter_nlm`:
>   - `patch_size`: 2D patch size (e.g. 3, 5, 7).
>   - `patch_distance`: Search radius (e.g. 4, 6, 8).
>   - `h`: Filtering parameter (smoothing strength).
> - Justify the best parameters and compare filtering performance against standard smoothing methods.


In [ ]:
# TODO: Non-Local Means parameter tuning and evaluation
# [Call filter_nlm(noisy_img, patch_size=..., patch_distance=..., h=...)]
# Remember to evaluate PSNR/SSIM and save figures: save_figure(fig, "05_nlm_evaluation.png")


## 4. Anisotropic Filtering: Perona and Malik
Anisotropic diffusion performs adaptive smoothing based on the local gradient magnitude $\|\nabla I\|$:
$$\frac{\partial I}{\partial t} = \text{div}(c(x, y, t) \nabla I)$$
- In homogeneous regions (low gradient): $c \approx 1 \rightarrow$ strong diffusion (Gaussian smoothing).
- Across edges and boundaries (high gradient): $c \approx 0 \rightarrow$ diffusion stops (edge preservation).

> **Assignment Requirement**:  
> - Evaluate with both images and both noise types at three intensity levels.
> - Tune parameters: number of iterations `n_iter`, conduction coefficient threshold `kappa`, and compare both conductivity functions (`option=1` exponential vs `option=2` rational).
> - Compare edge preservation performance against standard smoothing methods.


In [ ]:
# TODO: Anisotropic Diffusion (Perona & Malik) experiments
# [Call filter_perona_malik(noisy_img, n_iter=..., kappa=..., option=...)]
# Save figures: save_figure(fig, "06_perona_malik_evaluation.png")


## 5. Other Types of Advanced Filtering (State-of-the-Art)
Research other advanced filtering methods available in the scientific literature. Provide an explanation of the method, evaluate it with the same images and noise scenarios, and benchmark its performance against NLM, Perona-Malik, and standard methods.

Two candidate methods are provided in `src.filters_advanced`:
1. **Bilateral Filter (`filter_bilateral`)**: Combines spatial domain closeness and photometric range similarity.
2. **Total Variation Denoising (`filter_total_variation`)**: Variational regularizer (Rudin-Osher-Fatemi model via Chambolle's algorithm) preserving sharp interfaces.


In [ ]:
# TODO: Advanced Literature Filter experiment (Bilateral or Total Variation)
# [Call your chosen advanced filter here]
# Save figures: save_figure(fig, "07_advanced_sota_filter.png")


## 6. Global Comparison and Quantitative Metrics Summary
For the 10-page final report, consolidate the **PSNR** and **SSIM** values across all filtering methods and noise conditions into a structured table.


In [ ]:
# TODO: Build a summary DataFrame with all benchmarking results
# Example:
# summary_df = pd.DataFrame([
#     {"Image": "Brain MRI", "Noise": "Gaussian (Med)", "Filter": "NLM", "PSNR (dB)": 30.65, "SSIM": 0.683},
#     ...
# ])
# summary_df.to_csv("../results/tables/metrics_summary.csv", index=False)
# summary_df


## 7. Discussion and Report Guidelines (Max. 10 Pages)
When writing your PDF submission report, ensure you address the following key points:
1. **Gaussian vs. Impulse Noise**: Why does the median filter outperform linear filters under salt-and-pepper noise?
2. **Noise Reduction vs. Edge Blurring Trade-off**: Which filters preserve fine anatomical structures (e.g. brain gyri, bone edges) best?
3. **NLM vs. Anisotropic Diffusion**: How do patch-based redundancy and gradient-based diffusion compare in edge preservation and texture retention?
4. **Computational Complexity**: Discuss execution time and scalability differences between local filters, iterative diffusion, and non-local patch searches.
5. **Mandatory Report Requirements**: State the individual contribution of each team member, and disclose any use of AI tools (purpose and sections).
